## 数据检索
用户输入只是问题，答案需要从已知的数据库或者文档中获取。因此，我们需要使用检索器获取上下文，并通过“question”键下的用户输入。

### RAG
检索增强生成（Retrieval-augmented Generation，RAG），是当下最热门的大模型前沿技术之一。如果将“微调（finetune）”理解成大模型内化吸收知识的过程，那么RAG就相当于给大模型装上了“知识外挂”，基础大模型不用再训练即可随时调用特定领域知识。

In [1]:
from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

# embeddings_path = "D:\\ai\\download\\bge-large-zh-v1.5"
# embeddings_path = "/home/vincent/.lmstudio/models/bge-large-zh-v1.5"
embeddings_path = "/home/vincent/.lmstudio/models/bge-small-zh-v1.5"

embeddings = HuggingFaceEmbeddings(model_name=embeddings_path)

In [2]:
vectorstore = FAISS.from_texts(
    ["小明在华为工作","熊喜欢吃蜂蜜"],
    embedding=embeddings
)
vectorstore



/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [3]:
#使用向量数据库生成检索器
retriever = vectorstore.as_retriever()

In [4]:
retriever.invoke("熊喜欢吃什么？")

/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[Document(id='6d8d4c45-94e7-486e-b7ce-5322c81ca7ca', metadata={}, page_content='熊喜欢吃蜂蜜'),
 Document(id='74e0ad1f-f68d-4750-a3c2-27b49fb72595', metadata={}, page_content='小明在华为工作')]

In [5]:
from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)


In [6]:
from langchain_core.prompts import ChatPromptTemplate

template ="""
只根据以下文档回答问题：
{context}

问题：{question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [7]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

outputParser = StrOutputParser()

setup_and_retrieval = RunnableParallel(
    {
        "context":retriever,
        "question":RunnablePassthrough()
    }
)

chain = setup_and_retrieval | prompt | model | outputParser

In [9]:
chain.invoke("小明在哪里工作？")

/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


"<think>\n好的，我需要回答用户的问题：“小明在哪里工作？”根据提供的文档内容，首先我要仔细查看这两个文档的信息。\n\n第一个文档的ID是'74e0ad1f-f68d-4750-a3c2-27b49fb72595'，其metadata为空，page_content显示“小明在华为工作”。第二个文档的ID是'6d8d4c45-94e7-486e-b7ce-5322c81ca'，metadata同样为空，page_content是“熊喜欢吃蜂蜜”。\n\n问题问的是小明的工作地点，而根据第一个文档的内容，明确提到小明在华为工作。因此，答案应该是华为。需要确认是否有其他可能的信息被遗漏，但这两个文档看起来都是单独的页面内容，没有其他相关的信息。所以最终的答案应该是小明在华为工作。\n</think>\n\n小明在华为工作。"

In [10]:
#完整链式
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke("小明在哪里工作？")

/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


"<think>\n好的，我现在需要回答用户的问题：“小明在哪里工作？”根据提供的文档内容，我需要仔细查看这两个文档的信息。\n\n第一个文档的ID是'74e0ad1f-f68d-4750-a3c2-27b49fb72595'，其页面内容是“小明在华为工作”。第二个文档的ID是'6d8d4c45-94e7-486e-b7ce-5322c81ca'，页面内容是“熊喜欢吃蜂蜜”。\n\n问题问的是小明的工作地点，而这两个文档中只有第一个明确提到了小明在华为工作。第二个文档的内容与问题无关，只是描述了熊喜欢蜂蜜。因此，正确的答案应该是小明在华为工作。\n</think>\n\n根据提供的文档信息，小明在华为工作。"

In [11]:
from operator import itemgetter

template ="""
只根据以下文档回答问题：
{context}

问题：{question}
回答问题请加上称呼"{name}"。
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "name": itemgetter("name"),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke({"question":"小明在哪里工作？","name":"主人"})

/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


'<think>\n好的，我需要解决用户的问题：“小明在哪里工作？”并按照要求加上“主人”这个称呼来回答。首先，我要仔细查看提供的文档内容。\n\n在文档中，有两个文档。第一个是id为74e0ad1f-f68d-4750-a3c2-27b49fb72595的页面，其中提到“小明在华为工作”。第二个文档id为6d8d4c45-94e7-486e-b7ce-5322c81ca，内容是“熊喜欢吃蜂蜜”。\n\n用户的问题直接询问的是小明的工作地点，而根据文档中的信息，明确指出小明在华为工作。因此，答案应该是“主人，在华为工作”。需要确保回答中包含正确的称呼，并且准确引用文档中的信息。\n</think>\n\n主人，在华为工作'

In [12]:
import os
print(os.getcwd())

/mnt/data/ai课程/老陈-2024-Langchain大模型AI应用与多智能体实战开发/Langchain大模型AI应用实战开发-资料/01-Langchain


In [18]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader("/home/vincent/ai课程/老陈-2024-Langchain大模型AI应用与多智能体实战开发/Langchain大模型AI应用实战开发-资料/02-agent/txt",
                         loader_cls=TextLoader)

docs = loader.load()
docs

[Document(metadata={'source': '/home/vincent/ai课程/老陈-2024-Langchain大模型AI应用与多智能体实战开发/Langchain大模型AI应用实战开发-资料/02-agent/txt/faq-7923.txt'}, page_content='众测活动\n\n整体介绍：\n\n一、活动定义：众测是以低价试销的形式，通过收集评价、销量等方法，用于测试市场对新商品的反应，用于及时优化销售策略和引导商家改进。\n\n二、优势：众测价通常比较优惠，以不高于大促促销价为原则，最终以与物权方谈判结果为准。\n\n三、适用范围：华为商城所有产品。\n\n\n\n参与方式：\n\n1、华为商城众测的入口在哪里？\n\n华为商城搜索“众测”，即可看到众测入口，点击进入即可；\n\n2、众测上新频次：\n\n众测频道每周一至周五不定期更新上架，部分商品可能会特别调整；\n\n3、众测活动时间：\n\n一款产品众测时间通常为10-20天（热销的商品可能会延期5-10天）。\n\n \n\n常见问题：\n\n1、众测商品的来源?\n\n     您好，众测商品主要来源为华为商城上架的新品，热销的爆款商品也会不定期通过众测回归上线。\n\n2、众测商品的价格会优惠吗？\n\n     您好，众测商品价格对标618和双十一，一般都是该商品某段时间的最低促销价。\n\n3、 众测商品质量会不稳定么？\n\n     您好，众测商品也是量产的正品新品，质量与正式上架商品一致。 \n\n4、众测商品买下后多久发货？\n\n     您好，请以商品页显示为准。\n\n5、提交活动订单后多久内支付？\n\n     您好，提交订单后最长付款时效为24小时，逾期订单自动取消\n\n6、成功下单后怎么查询众测订单？\n\n     您好，成功下单后，您可以通过华为商城手机APP、华为商城手机WAP版、华为商城电脑网页版任意一端登录下单账户，在“我的订单”查询。（众测商品订单查询方式等同于正常商品订单）\n\n7、订单支付后未发货前可以取消订单吗？\n\n     您好，在发货前消费者可以取消订单。\n\n8、取消订单后多久内退回款项？\n\n     您好，和华为商城智能家居生态产品通用退款时效一致，3-5个工作日。\n\n9、一个账号可以参与几次众

In [19]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader(
    "/home/vincent/ai课程/老陈-2024-Langchain大模型AI应用与多智能体实战开发/Langchain大模型AI应用实战开发-资料/02-agent/txt",
    # glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)
docs = loader.load()
print(f"成功加载 {len(docs)} 个文档")
docs

100%|██████████| 3/3 [00:00<00:00, 336.98it/s]

成功加载 3 个文档


[Document(metadata={'source': '/home/vincent/ai课程/老陈-2024-Langchain大模型AI应用与多智能体实战开发/Langchain大模型AI应用实战开发-资料/02-agent/txt/faq-7923.txt'}, page_content='众测活动\n\n整体介绍：\n\n一、活动定义：众测是以低价试销的形式，通过收集评价、销量等方法，用于测试市场对新商品的反应，用于及时优化销售策略和引导商家改进。\n\n二、优势：众测价通常比较优惠，以不高于大促促销价为原则，最终以与物权方谈判结果为准。\n\n三、适用范围：华为商城所有产品。\n\n\n\n参与方式：\n\n1、华为商城众测的入口在哪里？\n\n华为商城搜索“众测”，即可看到众测入口，点击进入即可；\n\n2、众测上新频次：\n\n众测频道每周一至周五不定期更新上架，部分商品可能会特别调整；\n\n3、众测活动时间：\n\n一款产品众测时间通常为10-20天（热销的商品可能会延期5-10天）。\n\n \n\n常见问题：\n\n1、众测商品的来源?\n\n     您好，众测商品主要来源为华为商城上架的新品，热销的爆款商品也会不定期通过众测回归上线。\n\n2、众测商品的价格会优惠吗？\n\n     您好，众测商品价格对标618和双十一，一般都是该商品某段时间的最低促销价。\n\n3、 众测商品质量会不稳定么？\n\n     您好，众测商品也是量产的正品新品，质量与正式上架商品一致。 \n\n4、众测商品买下后多久发货？\n\n     您好，请以商品页显示为准。\n\n5、提交活动订单后多久内支付？\n\n     您好，提交订单后最长付款时效为24小时，逾期订单自动取消\n\n6、成功下单后怎么查询众测订单？\n\n     您好，成功下单后，您可以通过华为商城手机APP、华为商城手机WAP版、华为商城电脑网页版任意一端登录下单账户，在“我的订单”查询。（众测商品订单查询方式等同于正常商品订单）\n\n7、订单支付后未发货前可以取消订单吗？\n\n     您好，在发货前消费者可以取消订单。\n\n8、取消订单后多久内退回款项？\n\n     您好，和华为商城智能家居生态产品通用退款时效一致，3-5个工作日。\n\n9、一个账号可以参与几次众